# Week 1 — Research Question and Provisional Lane

**Intern:** Sathwik | **Track:** Machine Learning | **Assignment:** ML-02

## 1. My lane and why

I'm choosing **Lane 2: Refresh / Content Opportunity Scoring**.

This lane asks a question that's easy to explain and easy to act on: *which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?* I picked it over the other three core lanes because:

- It maps directly onto a real workflow a content or SEO team already runs (a prioritized review queue), so the output is immediately usable rather than purely exploratory.
- The starter dataset (`data/raw/content_refresh_anonymized.csv`) is built exactly for this lane — it already has the fields needed to build a baseline score and compare it against a learned model (search volume, position, CTR, engagement, freshness, trend direction).
- The lane guide's starter pipeline (`scripts/01`–`05`) gives me a working baseline-vs-model comparison out of the box (baseline ROC AUC 0.627 vs random forest 0.750), which gives me a concrete, honest benchmark to try to beat or at least understand over the next 7 weeks — rather than starting from nothing.
- Compared to the advanced lanes (AI Referral Opportunity, Growth/Recovery Prediction), this one has denser, more reliable signal in the data (GSC impressions/clicks are far less sparse than AI-session rows), which keeps the project honest and low-risk for a first capstone.

## 2. The question: decision, action, and cost of a wrong call

**Decision this informs:** Which existing content pages a content/SEO reviewer should look at *first*, out of a much larger inventory, when they only have capacity to review a limited number per week.

**Action someone takes:** A reviewer opens the top-ranked pages from the queue and decides, page by page, whether to refresh (update content/metadata), expand (add depth), protect (leave alone, monitor), or prune (remove/consolidate) — using the reason codes attached to each score as their starting evidence.

**Cost of a wrong recommendation:**
- *False positive* (page flagged as "review now" but it was actually fine): wastes a reviewer's limited time — the real cost is opportunity cost, since that hour wasn't spent on a page that genuinely needed attention.
- *False negative* (a page that's actually declining or has real opportunity, but the queue never surfaces it): the page keeps losing visibility/traffic silently until someone notices by chance, which is the more expensive failure mode since it compounds over time.
- Because reviewer time is the scarce resource, precision in the top of the queue (precision@20 / precision@50) matters more than perfect recall across the whole inventory — this is why the lane guide emphasizes top-K metrics over generic accuracy.

## 3. Quick look at the data

Loading the starter dataset and pulling a few real numbers to confirm this lane has usable signal before committing further.

In [ ]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

print(f"Total rows: {len(df):,}")
print(f"Columns: {list(df.columns)}\n")

# Apply the same filters the starter pipeline uses (impressions_90d > 0, content_age_days >= 90)
usable = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)]
print(f"Rows usable after starter filters: {len(usable):,} ({len(usable)/len(df):.1%} of total)")

# How many pages currently show a declining trend? (the starter proxy label)
declining_pct = (usable['trend_direction'] == 'down').mean()
print(f"Share of usable pages currently trending down: {declining_pct:.1%}")

# Median impressions for declining vs non-declining pages, as a quick sanity check on signal
median_impr_declining = usable.loc[usable['trend_direction'] == 'down', 'impressions_90d'].median()
median_impr_other = usable.loc[usable['trend_direction'] != 'down', 'impressions_90d'].median()
print(f"Median 90d impressions — declining pages: {median_impr_declining:.0f} | other pages: {median_impr_other:.0f}")

*(Run the cell above — it will print the real row counts, decline share, and impression medians from your copy of the starter dataset. Those printed numbers are your 2–3 supporting numbers for this section; no need to retype them elsewhere, the executed cell output is the evidence.)*

## 4. Careful words: what I can and can't claim

**What I can claim:**
- The starter dataset shows measurable variation in visibility, engagement, and freshness across content, and a simple rule-based baseline already separates declining from non-declining pages better than chance (baseline ROC AUC ~0.63 per the pipeline's committed report).
- A ranked queue built from this data can help a reviewer prioritize *where to look first*, given limited review capacity.

**What I can't claim:**
- I can't claim that refreshing a flagged page will *cause* recovery — that requires a controlled experiment, which this observational data doesn't provide.
- I can't claim I've identified a Google algorithm factor — only associations between observable content/search signals and outcomes.
- The current label (`trend_direction == "down"`) is a present-window bucket, not a future outcome, so early results describe pattern association, not prediction — I plan to move toward a future-window label (prior 90 days → next 30 days) as the capstone matures, per the lane guide's guidance.
- Any result from the 30,000-row starter slice is not a benchmark on the full warehouse and will need to be re-validated at scale.

## 5. Self-check

- [x] Picks one of the four predefined lanes (Lane 2: Refresh / Content Opportunity Scoring) with a stated reason.
- [x] Names the decision (what to review first) and the action (refresh/expand/protect/prune).
- [x] Shows at least two real numbers pulled directly from the starter dataset via an executed code cell (row counts, decline share, impression medians).
- [x] Explains why this is not just "train a model" — it's tied to a reviewer's real capacity constraint and a defined cost for getting it wrong.
- [x] Uses careful language — no causal claims, no algorithm-factor claims, labels honesty about the proxy label limitation.

This lane choice can be revisited until the end of Week 4 if the data doesn't hold up as expected.